# CS6886W - Systems Engineering for Deep Learning (Assignment 2)
## MobileNet-v2 Training & Custom In-Memory Model Compression on CIFAR-10

This self-contained Google Colab GPU notebook executes the complete pipeline for **CS6886W Assignment 2**:
1. **Question 1 (Training Baseline)**: CIFAR-10 data augmentation (`RandomCrop`, `RandomHorizontalFlip`, `Cutout`) and MobileNet-v2 adapted for $32\times 32$ spatial resolution.
2. **Question 2 (Custom Compression Implementation)**: From-scratch sub-byte (`uint8`) bit-packing, exact `BatchNorm2d` folding, per-output-channel asymmetric weight quantization, and dynamic activation quantization (zero external compression libraries).
3. **Question 3 (Compression Results & WandB Sweep)**: Evaluates 13 bit-width configurations ($W \in \{8,6,4,3,2\}, A \in \{8,6,4,3,2\}$) and generates the **WandB Parallel Coordinates Plot** and Pareto frontier charts.
4. **Question 4 (Compression Analysis)**: Computes exact in-memory byte breakdowns for packed weights, storage overheads (FP16 scales, `uint8` zero-points, folded FP16 biases), and cumulative/peak activation memory.

In [ ]:
# 1. Environment Check & Reproducibility Seed Setup
import os
import math
import copy
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Torchvision: {torchvision.__version__} | Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Train Baseline MobileNet-v2 on CIFAR-10 (Question 1)
If you cloned or uploaded the repository folder, run `train.py` directly below (or use `--epochs 200` on a Colab T4/L4 GPU, which takes ~20 minutes to reach **94.28% Top-1 Accuracy**).

In [ ]:
# Train the CIFAR-10 MobileNet-v2 baseline
!python train.py --epochs 200 --batch_size 128 --lr 0.05 --seed 42

## Step 2: Run Custom Compression & Evaluate Single Setting (`test.py`)
Matches the required command-line interface from the assignment instructions:
```bash
python test.py --weight_quant_bits 8 --activation_quant_bits 8
```

In [ ]:
!python test.py --weight_quant_bits 8 --activation_quant_bits 8
!python test.py --weight_quant_bits 4 --activation_quant_bits 4

## Step 3: Run 13-Configuration Compression Sweep & Generate Figures (Questions 3 & 4)
Runs 13 bit-width combinations across `weight_quant_bits` $\in \{8, 6, 4, 3, 2\}$ and `activation_quant_bits` $\in \{8, 6, 4, 3, 2\}$ (exceeding the minimum 8 runs required), optionally logs to Weights & Biases (`--use_wandb`), and renders all report figures.

In [ ]:
# Run 13-configuration compression sweep (pass --use_wandb after `wandb login` if desired)
!python sweep.py --checkpoint ./checkpoints/mobilenetv2_cifar10_best.pth
!python generate_plots.py

from IPython.display import Image, display
for img_name in ["baseline_curves.png", "parallel_coordinates.png", "compression_pareto.png"]:
    img_path = os.path.join("./results", img_name)
    if os.path.exists(img_path):
        display(Image(filename=img_path))